<a href="https://colab.research.google.com/github/joe6302413/Microelectronics-demo/blob/main/Transistor_DC_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transistors at DC — Interactive Lab
GUPS2001 Electronics I. Run the two cells below in order, then click the `trycloudflare.com` link printed by the second cell.

In [ ]:
# 1. Clone the project and enter its folder
!git clone -q https://github.com/joe6302413/Microelectronics-demo.git
%cd Microelectronics-demo

# 2. Install uv and put it on the PATH (%env keeps it for every later cell)
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
%env PATH=/root/.local/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin

# 3. Install cloudflared (skipped if it is already there)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 4. Install the Python packages now, so Streamlit starts quickly in the next cell
print("📦 Installing the packages for the transistor lab (NumPy, Streamlit, schemdraw, ...)")
!uv sync -q

print("✅ Setup finished. Now run the next cell.")

In [ ]:
import os
import time
import urllib.request

# 1. Start Streamlit in the background
# --server.address=127.0.0.1 works best with the Cloudflare tunnel; websocket compression
# makes every slider update smaller, which helps through the tunnel
print("🚀 Starting the Streamlit app ...")
os.system("uv run streamlit run transistor.py --server.port 8501 --server.address=127.0.0.1 "
          "--server.headless=true --server.fileWatcherType=none --server.enableWebsocketCompression=true "
          "--browser.gatherUsageStats=false > streamlit.log 2>&1 &")

# 2. Wait until the app answers (instead of a fixed delay)
for _ in range(60):
    try:
        if urllib.request.urlopen("http://127.0.0.1:8501/_stcore/health", timeout=1).read() == b"ok":
            break
    except Exception:
        time.sleep(0.5)

# 3. Open the Cloudflare tunnel and print the link
print("🔗 Opening a secure tunnel. Click the trycloudflare.com link below:")
print("-" * 50)
!cloudflared tunnel --url http://127.0.0.1:8501 2>&1 | grep -o 'https://.*\.trycloudflare\.com'
print("-" * 50)